# Phase 6 SMO / PMWO MVP

Study-grade source-mask optimization and PMWO candidate sweeps over mask duty, source shape, dose, pupil, wavefront, and Phase 5 LWR penalty.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().resolve()
if not (repo / "src").exists() and (repo.parent / "src").exists():
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

In [ ]:
from src import constants as C
from src.illuminator import annular_source, point_source
from src.mask import MaskGrid, line_space_pattern
from src.pmwo import opc_bias_mask_candidates, pmwo_grid_search, pupil_obscuration_candidates
from src.pupil import PupilSpec
from src.smo import SMOObjectiveWeights, fast_smo_grid_search, line_space_mask_candidates

In [ ]:
grid = MaskGrid(nx=256, ny=64, pixel_size=2e-9)
pitch_m = 40e-9
target_mask = line_space_pattern(grid, pitch_m=pitch_m, duty_cycle=0.5)
target_printed = target_mask == 0.0

mask_candidates = line_space_mask_candidates(grid, pitch_m, [0.35, 0.45, 0.50, 0.55, 0.65])
sources = (
    point_source(name="point"),
    annular_source(0.2, 0.7, num_radial=1, num_azimuthal=4, name="annular"),
)
pupil = PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.0)
weights = SMOObjectiveWeights(cd=2.0, epe=1.0, lwr=0.25)
pmwo_masks = opc_bias_mask_candidates(grid, pitch_m, [-4e-9, 0.0, 4e-9])
pupil_candidates = pupil_obscuration_candidates(pupil, [0.8, 0.0])

In [ ]:
pmwo_result = pmwo_grid_search(
    target_printed,
    grid,
    pmwo_masks,
    [sources[1]],
    pupil_candidates,
    [1.0],
    weights=SMOObjectiveWeights(cd=2.0, epe=1.0, lwr=0.0),
)

pmwo_best = pmwo_result.best
{
    "initial_loss": pmwo_result.initial.metrics.loss,
    "best_loss": pmwo_best.metrics.loss,
    "improvement_fraction": pmwo_result.improvement_fraction,
    "best_mask": pmwo_best.mask_candidate.name,
    "best_pupil": pmwo_best.pupil_candidate.name,
    "mean_2d_epe_nm": pmwo_best.epe_map.mean_abs_epe_m * 1e9,
}

In [ ]:
result = fast_smo_grid_search(
    target_printed,
    grid,
    mask_candidates,
    sources,
    [0.9, 1.0, 1.1],
    pupil_spec=pupil,
    weights=weights,
)

summary_rows = [
    {
        "mask": item.mask_candidate.name,
        "source": item.source_shape.name,
        "dose": item.dose,
        "loss": item.metrics.loss,
        "cd_error": item.metrics.cd_error_fraction,
        "epe_error": item.metrics.epe_error_fraction,
        "lwr_fraction": item.metrics.lwr_fraction,
    }
    for item in result.history
]

best = result.best
{
    "initial_loss": result.initial.metrics.loss,
    "best_loss": best.metrics.loss,
    "improvement_fraction": result.improvement_fraction,
    "best_mask": best.mask_candidate.name,
    "best_source": best.source_shape.name,
    "best_dose": best.dose,
}

In [ ]:
top_rows = sorted(summary_rows, key=lambda row: row["loss"])[:8]
labels = [f"{row['mask'].split('_')[-1]}\n{row['source']}\nD={row['dose']:.1f}" for row in top_rows]
losses = [row["loss"] for row in top_rows]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].bar(labels, losses, color="#4C78A8")
axes[0].set_ylabel("weighted loss")
axes[0].set_title("Best SMO candidates")
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(axis="y", alpha=0.25)

cy = best.wafer_grid.ny // 2
x_nm = (np.arange(best.wafer_grid.nx) - best.wafer_grid.nx // 2) * best.wafer_grid.pixel_x_m * 1e9
axes[1].plot(x_nm, target_printed[cy, :].astype(float), label="target", linewidth=2)
axes[1].plot(x_nm, best.printed[cy, :].astype(float), label="best printed", linestyle="--")
axes[1].set_xlabel("x at wafer (nm)")
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title("Target vs optimized print")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.show()